# 12.1 TensorFlow快速浏览


---

## 一、本章核心背景
从第1-11章主要使用 **tf.keras**（高层API）构建模型，虽然覆盖95%的常规需求（回归、分类、宽与深层网络、正则化、调参），但**仍有特殊场景无法覆盖**：
- 需要额外的**损失函数控制**
- 需要**自定义指标、模型、初始化程序**
- 需要**定制训练循环**（例如对梯度使用特殊约束）
- 需要**网络不同部分使用多个优化器**

本章将深入探讨 **TensorFlow 底层 Python API** 与**自动微分**机制，实现更灵活的自定义模型与训练算法。

---

## 二、TensorFlow 基础概览
### 1. 版本与定位
- **TensorFlow 2.0 (beta)** 于2019年6月发布，相比TF 1.x大幅简化，更易于使用。
- 核心定位：**强大的数值计算库**，特别适合**大规模机器学习**与并行计算（GPU/TPU）。

### 2. 核心特性（为什么它是主流？）
| 特性 | 解释                                                                        |
|:--- |:--------------------------------------------------------------------------|
| **多后端/多设备支持** | 完美适配 CPU、GPU、TPU（张量处理单元），计算速度极快                                           |
| **跨平台** | 支持 Windows、Linux、macOS，还可在移动端（TensorFlow Lite）、浏览器（TensorFlow.js）、嵌入式设备运行 |
| **自动微分 (Autograd)** | 自动计算梯度（反向传播），这是神经网络训练的核心，不用手动推导公式                                         |
| **生态系统完善** | 包含数据加载、预处理、可视化（TensorBoard）、模型优化、部署等一整套工具链                                |

---

## 三、TensorFlow 的 Python API 体系
TensorFlow API 分为**底层**与**高层**，95%的场景只用高层API，其余5%需用底层API做定制：

### 1. 高层 API（直接用即可）
| 模块 | 功能 |
|:--- |:--- |
| **tf.keras** | 构建、训练、评估模型（本书前11章主要使用） |
| **tf.estimator** | 高级深度学习API（团队建议少用，优先tf.keras） |
| **tf.distribute** | 分布式计算，多GPU/多主机训练 |
| **tf.data** | 高效数据加载、预处理流水线 |
| **tf.image/audio/signal** | 图像、音频、信号处理 |

### 2. 底层 API（灵活定制用）
| 模块 | 功能 |
|:--- |:--- |
| **tf.GradientTape** | **自动微分核心**：记录计算过程，自动求梯度（本章重点） |
| **tf.function** | 将Python代码编译为图执行，加速运行 |
| **tf.keras.layers** | 自定义层、自定义模型 |
| **tf.losses/metrics/optimizers** | 自定义损失、指标、优化器 |

### 3. 其他核心模块
- **可视化**：TensorBoard（tf.summary）
- **数据结构**：tf.ragged（稀疏张量）、tf.sparse（稀疏张量）
- **数学运算**：tf.linalg（线性代数）、tf.random（随机数）等
- **部署优化**：tf.tpu（底层优化）、tf.xla（JIT编译加速）

---

## 四、TensorFlow 的架构
TensorFlow 运行机制像**分层执行引擎**：

1. **Python 前端**：
   - 用户代码（Keras / 底层API）
   - 95% 使用 Keras，5% 使用低阶 Python API（ops）

2. **执行引擎**：
   - 自动图执行（AutoGraph）
   - 本地/分布式执行引擎

3. **底层内核（C++ 实现）**：
   - CPU kernels、GPU kernels、TPU kernels
   - 负责具体的数值计算，速度极快

**作用**：不管是Python代码还是其他语言（C++、Go、Java），最终都交给高效的C++内核处理，实现跨语言、跨平台运行。

---

## 五、生态系统与资源
TensorFlow 不仅是库，更是**庞大生态系统核心**：
1. **TensorBoard**：可视化训练过程（损失、准确率、结构图等）
2. **TensorFlow Extended (TFX)**：端到端生产环境套件（数据验证、预处理、分析、部署）
3. **TensorFlow Hub**：预训练模型仓库，可直接下载复用（如BERT、ResNet等）
4. **社区资源**：
   - GitHub 项目库：[github.com/tensorflow/tensorflow](sslocal://flow/file_open?url=https%3A%2F%2Fgithub.com%2Ftensorflow%2Ftensorflow&flow_extra=eyJsaW5rX3R5cGUiOiJjb2RlX2ludGVycHJldGVyIn0=)
   - 第三方资源：[github.com/joelowery/awesome-tensorflow](sslocal://flow/file_open?url=https%3A%2F%2Fgithub.com%2Fjoelowery%2Fawesome-tensorflow&flow_extra=eyJsaW5rX3R5cGUiOiJjb2RlX2ludGVycHJldGVyIn0=)
   - ML 论文代码平台：[paperswithcode.com](sslocal://flow/file_open?url=https%3A%2F%2Fpaperswithcode.com%2F&flow_extra=eyJsaW5rX3R5cGUiOiJjb2RlX2ludGVycHJldGVyIn0=)（可直接找实现代码）

---

## 六、社区与支持
- **团队**：Google Brain 团队 + 全球社区（stackoverflow、GitHub）
- **贡献方式**：提Bug、提交功能请求、参与讨论（Google 讨论组）

---


# 12.2 像Numpy一样使用TensorFlow

## 12.2.1 张量和操作

In [2]:
import numpy as np
# 创建张量
import tensorflow as tf

from Chapter10_Introduction_of_Keras_Artificial_Neural_Network.ch10_Introduction_of_Keras_artificial_neural_network import \
    keras_reg

from Chapter03_Classification.ch03_classification import precisions
from keras.src.legacy.backend import gradients

from Chapter11_Training_Deep_Neural_Networks.ch11_Training_deep_neural_networks import optimizer

tf.constant([[1.,2.,3.],[4.,5.,6.]])

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)>

In [3]:
tf.constant(42)

<tf.Tensor: shape=(), dtype=int32, numpy=42>

In [4]:
# 查看形状和数据类型
t = tf.constant([[1.,2.,3.],[4.,5.,6.]])
t.shape

TensorShape([2, 3])

In [5]:
t.dtype

tf.float32

In [6]:
t[:,1:]

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[2., 3.],
       [5., 6.]], dtype=float32)>

In [7]:
t[...,1,tf.newaxis]

<tf.Tensor: shape=(2, 1), dtype=float32, numpy=
array([[2.],
       [5.]], dtype=float32)>

In [8]:
# 还可以有各种张量操作
t+10

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[11., 12., 13.],
       [14., 15., 16.]], dtype=float32)>

In [9]:
tf.square(t)

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[ 1.,  4.,  9.],
       [16., 25., 36.]], dtype=float32)>

In [10]:
# 乘法
t @ tf.transpose(t)


<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[14., 32.],
       [32., 77.]], dtype=float32)>


---

## 一、核心总览
TensorFlow的核心是**张量（Tensor）**，所有计算都围绕张量的流动展开。
张量本质上就是**多维数值数组**，和NumPy的`ndarray`高度相似，但额外支持GPU加速、自动微分、分布式计算等深度学习必备能力。
本节的目标是：用你熟悉的NumPy思维，快速上手TensorFlow的张量操作。

---

## 二、张量基础：创建与属性
### 1. 张量的创建
最常用的创建方法是`tf.constant()`，支持标量、向量、矩阵、高维数组：
```python
import tensorflow as tf
import numpy as np

# 1. 创建2行3列的浮点矩阵
matrix = tf.constant([[1., 2., 3.], [4., 5., 6.]])
print(matrix)
# <tf.Tensor: shape=(2, 3), dtype=float32, numpy=array([[1., 2., 3.], [4., 5., 6.]], dtype=float32)>

# 2. 创建标量（单个数值）
scalar = tf.constant(42)
print(scalar)
# <tf.Tensor: id=1, shape=(), dtype=int32, numpy=42>
```

### 2. 张量的核心属性
和NumPy的`ndarray`一样，张量有两个关键属性：
| 属性 | 作用 | 示例 |
|------|------|------|
| `shape` | 张量的维度/形状 | `matrix.shape` → `TensorShape([2, 3])` |
| `dtype` | 张量的数据类型 | `matrix.dtype` → `tf.float32` |

### 3. 张量与NumPy的互转
- **张量 → NumPy数组**：直接用`.numpy()`方法
  ```python
  arr = matrix.numpy()  # 得到np.ndarray
  ```
- **NumPy数组 → 张量**：直接用`tf.constant()`
  ```python
  t = tf.constant(arr)  # 得到tf.Tensor
  ```

---

## 三、张量索引：完全兼容NumPy语法
张量的索引、切片规则**100%和NumPy一致**，上手零成本：
```python
t = tf.constant([[1., 2., 3.], [4., 5., 6.]])

# 1. 取第1行（索引从0开始）
print(t[0])  # [1., 2., 3.]

# 2. 取第1列（所有行，第1列）
print(t[:, 1])  # [2., 5.]

# 3. 取第2行，第2-3列
print(t[1, 1:])  # [5., 6.]
```

---

## 四、张量运算：NumPy式操作
TensorFlow实现了几乎所有NumPy的元素级运算，同时支持广播机制，用法完全一致：
### 1. 基础算术运算
```python
t = tf.constant([[1., 2., 3.], [4., 5., 6.]])

# 元素级加法
print(t + 10)  # 每个元素+10
# [[11., 12., 13.], [14., 15., 16.]]

# 元素级乘法
print(t * 2)  # 每个元素×2
# [[2., 4., 6.], [8., 10., 12.]]

# 矩阵乘法（@运算符，和NumPy一致）
print(t @ tf.transpose(t))  # 矩阵×矩阵的转置
```

### 2. 常用数学函数
```python
# 平方根
print(tf.sqrt(t))

# 指数运算
print(tf.exp(t))

# 求和、均值、最大值
print(tf.reduce_sum(t))  # 所有元素求和
print(tf.reduce_mean(t))  # 所有元素求均值
print(tf.reduce_max(t))  # 所有元素求最大值
```

### 3. 广播机制
和NumPy完全一致：不同形状的张量运算时，会自动扩展维度匹配：
```python
# 2×3矩阵 + 1×3向量 → 向量自动扩展为2×3，逐行相加
t1 = tf.constant([[1., 2., 3.], [4., 5., 6.]])
t2 = tf.constant([[10., 20., 30.]])
print(t1 + t2)
# [[11., 22., 33.], [14., 25., 36.]]
```

---

## 五、张量的类型：常用张量分类
除了最基础的`tf.Tensor`，TensorFlow还提供了多种特殊张量，适配不同场景：
| 张量类型 | 特点 | 适用场景 |
|----------|------|----------|
| **tf.constant** | 不可变张量（创建后不能修改） | 存储固定数据、常量 |
| **tf.Variable** | 可变张量（可修改值） | 存储模型权重、可训练参数 |
| **tf.RaggedTensor** | 不规则张量（每行长度不同） | 变长序列、文本数据 |
| **tf.SparseTensor** | 稀疏张量（仅存储非零元素） | 高维稀疏数据（如推荐系统） |
| **tf.TensorArray** | 动态张量数组 | 动态长度的张量序列 |

### 重点：tf.Variable（模型权重的核心）
`tf.Variable`是唯一可修改的张量，是神经网络权重的载体：
```python
v = tf.Variable([[1., 2., 3.], [4., 5., 6.]])
print(v)
# <tf.Variable 'Variable:0' shape=(2, 3) dtype=float32, numpy=array([[1., 2., 3.], [4., 5., 6.]], dtype=float32)>

# 修改值（assign方法）
v.assign(v * 2)  # 所有元素×2
v[0, 1].assign(99)  # 修改单个元素
```

---

## 六、其他核心操作
### 1. 类型转换
```python
# 转换数据类型
t = tf.constant([[1, 2], [3, 4]])  # int32
t_float = tf.cast(t, tf.float32)  # 转为float32
```

### 2. 张量形状操作
```python
# 重塑形状（reshape，和NumPy一致）
t = tf.constant([[1., 2., 3.], [4., 5., 6.]])
t_reshaped = tf.reshape(t, (3, 2))  # 3行2列
```

### 3. 随机张量
```python
# 生成随机张量（用于权重初始化、数据增强）
tf.random.normal(shape=(2, 3), mean=0, stddev=1)  # 正态分布
tf.random.uniform(shape=(2, 3), minval=0, maxval=1)  # 均匀分布
```

---

## 七、核心总结：TensorFlow vs NumPy
| 特性 | TensorFlow张量 | NumPy数组 |
|------|----------------|------------|
| 运行环境 | 支持CPU/GPU/TPU，分布式计算 | 仅CPU，单机 |
| 自动微分 | 支持（训练神经网络的核心） | 不支持 |
| 图执行 | 支持tf.function编译加速 | 仅即时执行 |
| 数据类型 | 丰富的特殊张量（可变、稀疏、不规则） | 仅基础ndarray |
| 生态 | 深度学习全栈工具链（Keras、部署、可视化） | 通用数值计算 |

**一句话核心区别**：NumPy是通用数值计算工具，TensorFlow是**专为深度学习设计的、带GPU加速和自动微分的NumPy增强版**。

---

## 八、后续内容预告
本节是TensorFlow底层API的基础，后续章节将基于张量操作，深入讲解：
1.  **自动微分（tf.GradientTape）**：实现自定义梯度计算
2.  **tf.function**：将Python代码编译为图，加速运行
3.  **自定义训练循环**：完全掌控模型训练的每一步
4.  **自定义层、自定义损失函数**：实现复杂的网络结构

---


# 12.2.2 张量和Numpy

In [12]:
import numpy as np
# 张量可以和Numpy配合使用
a = np.array([2.,4.,5.])
tf.constant(a)

<tf.Tensor: shape=(3,), dtype=float64, numpy=array([2., 4., 5.])>

In [13]:
t.numpy()

array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)

In [14]:
np.array(t)

array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)

In [15]:
tf.square(a)

<tf.Tensor: shape=(3,), dtype=float64, numpy=array([ 4., 16., 25.])>

In [16]:
np.square(t)

array([[ 1.,  4.,  9.],
       [16., 25., 36.]], dtype=float32)

注意，默认情况下Numpy使用64位精度，而TensorFlow使用32位精度

## 12.2.3 类型转换

In [17]:
# TensorFlow不会执行任何类型转换：如果对不兼容类型的张量执行操作，会引发异常
tf.constant(2.)+tf.constant(40)

InvalidArgumentError: cannot compute AddV2 as input #1(zero-based) was expected to be a float tensor but is a int32 tensor [Op:AddV2] name: 

In [18]:
tf.constant(2.)+tf.constant(40.,dtype = tf.float64)

InvalidArgumentError: cannot compute AddV2 as input #1(zero-based) was expected to be a float tensor but is a double tensor [Op:AddV2] name: 

In [20]:
# 当确实需要类型转换时，可以使用tf.cast()
t2 = tf.constant(40.,dtype = tf.float64)

In [21]:
tf.constant(2.)+tf.cast(t2,tf.float32)

<tf.Tensor: shape=(), dtype=float32, numpy=42.0>

## 12.2.4 变量

In [22]:
v = tf.Variable([[1.,2.,3.],[4.,5.,6.]])
v

<tf.Variable 'Variable:0' shape=(2, 3) dtype=float32, numpy=
array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)>

In [23]:
v.assign(2*v)

<tf.Variable 'UnreadVariable' shape=(2, 3) dtype=float32, numpy=
array([[ 2.,  4.,  6.],
       [ 8., 10., 12.]], dtype=float32)>

In [24]:
v[0,1].assign(42)

<tf.Variable 'UnreadVariable' shape=(2, 3) dtype=float32, numpy=
array([[ 2., 42.,  6.],
       [ 8., 10., 12.]], dtype=float32)>

In [25]:
v[:,2].assign([0.,1.])

<tf.Variable 'UnreadVariable' shape=(2, 3) dtype=float32, numpy=
array([[ 2., 42.,  0.],
       [ 8., 10.,  1.]], dtype=float32)>

In [30]:
v.scatter_nd_update(indices = [[0,0],[1,2]],updates = [100.,200.])

<tf.Variable 'UnreadVariable' shape=(2, 3) dtype=float32, numpy=
array([[100.,  42.,   0.],
       [  8.,  10., 200.]], dtype=float32)>

## 12.2.5 其他数据结构


---
## 一、核心数据结构分类与说明
### 1. 稀疏张量（`tf.SparseTensor`）
- **作用**：高效存储**包含大量零值**的张量，避免冗余存储
- **配套工具**：`tf.sparse` 程序包，提供稀疏张量的专属操作方法
- **适用场景**：高维稀疏数据（如推荐系统用户-物品交互矩阵、NLP 词袋特征）

### 2. 张量数组（`tf.TensorArray`）
- **本质**：张量的列表容器
- **特性**：
  - 默认大小固定，支持动态设置大小
  - 容器内所有张量必须**形状相同、数据类型一致**
- **适用场景**：动态长度的张量序列（如 RNN 时间步输出的存储）

### 3. 不规则张量（`tf.RaggedTensor`）
- **本质**：张量列表的静态容器
- **特性**：列表内每个张量**形状、数据类型一致**，但整体维度不规整（如不同长度的句子序列）
- **配套工具**：`tf.ragged` 程序包，提供不规则张量的操作方法
- **适用场景**：变长序列数据（如自然语言处理中不同长度的句子、CV 中不同数量的目标框）

### 4. 字符串张量
- **基础类型**：`tf.string` 类型的常规张量
- **核心特性**：
  - 存储**字节字符串**（非 Unicode 字符串），Python 3 中 Unicode 字符串会自动编码为 UTF-8 字节
  - `tf.string` 是原子级类型：字符串长度**不会出现在张量的形状中**；转换为 `tf.int32` 类型的 Unicode 代码点张量后，长度才会显示在形状中
- **配套工具**：`tf.strings` 程序包，支持字节字符串与 Unicode 字符串的互转、字符串操作
- **示例**：
  - 字节字符串：`b"caf\x3\xc\xa9"`（UTF-8 编码）
  - Unicode 代码点张量：`[99, 97, 102, 233]`（每个元素对应一个 Unicode 字符）

### 5. 集合
- **本质**：用常规张量（或稀疏张量）表示集合
- **表示规则**：集合由张量**最后一个轴上的向量**表示，例如 `tf.constant([[[1, 2], [3, 4]]])` 代表集合 `{1,2}` 和 `{3,4}`
- **配套工具**：`tf.sets` 程序包，提供集合的交、并、差等操作

### 6. 队列
- **作用**：跨多个步骤存储张量，用于异步数据处理、流水线调度
- **常见队列类型**（均在 `tf.queue` 包中）：
  | 队列类型 | 核心特性 | 适用场景 |
  |---|---|---|
  | `FIFOQueue` | 先进先出队列 | 普通顺序数据流水线 |
  | `PriorityQueue` | 支持元素优先级区分 | 按优先级调度任务 |
  | `RandomShuffleQueue` | 随机排序元素 | 训练数据打乱、随机采样 |
  | `PaddingFIFOQueue` | 支持填充，处理不同形状元素 | 批处理变长序列数据 |

---
## 二、总结
掌握**张量-运算-变量**的基础，再结合上述特殊数据结构，就可以在 TensorFlow 中实现自定义模型和训练算法，适配各类复杂数据场景（稀疏、变长、异步等）。

---

# 12.3 定制模型和训练算法

## 12.3.1 自定义损失函数

In [100]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
X_train = np.random.rand(1000, 10)
y_train = 3 * X_train[:, 0] + 2 * X_train[:, 1] - 1.5 * X_train[:, 2] + np.random.randn(1000) * 0.1
X_train_scaled = StandardScaler().fit_transform(X_train)
# 添加一些异常值
y_train[:20] += 10

X_test = np.random.rand(200, 10)
y_test = 3 * X_test[:, 0] + 2 * X_test[:, 1] - 1.5 * X_test[:, 2] + np.random.randn(200) * 0.1


In [35]:
# 手动创建Huber损失
def huber_fn(y_true, y_pred):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < 1
    squared_loss = tf.square(error)/2
    linear_loss = tf.abs(error)-0.5
    return tf.where(is_small_error,squared_loss,linear_loss)

In [36]:

# 3. 构建一个简单的回归模型
model = keras.models.Sequential([
    keras.layers.Dense(30, activation='relu', input_shape=(10,)),
    keras.layers.Dense(20, activation='relu'),
    keras.layers.Dense(1)
])

# 4. 编译模型（使用自定义损失）
model.compile(loss=huber_fn, optimizer='nadam')

# 5. 训练模型
history = model.fit(X_train, y_train, epochs=20, batch_size=32,
                    validation_data=(X_test, y_test), verbose=1)

# 6. 保存模型
model.save('my_model_with_a_custom_loss.h5')

Epoch 1/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 1.3251 - val_loss: 0.8507
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.7757 - val_loss: 0.4485
Epoch 3/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6060 - val_loss: 0.3712
Epoch 4/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4887 - val_loss: 0.2318
Epoch 5/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.3399 - val_loss: 0.0920
Epoch 6/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2358 - val_loss: 0.0251
Epoch 7/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2057 - val_loss: 0.0125
Epoch 8/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2005 - val_loss: 0.0102
Epoch 9/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1996 - val_loss: 0.0095
Epoch 10/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1990 - val_loss: 0.0090
Epoch 11/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1984 - val_loss: 0.0086
Epoch 12/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1982 - val_lo

## 12.3.2 保存和加载包含自定义组件的模型

In [38]:
# 加载模型
model = keras.models.load_model('my_model_with_a_custom_loss.h5', custom_objects={'huber_fn':huber_fn})

In [40]:
# 一个不同的阈值
def create_huber(threshold = 1.0):
    def huber_fn(y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < threshold
        squared_loss = tf.square(error)/2
        linear_loss = threshold*tf.abs(error)-threshold**2/2
        return tf.where(is_small_error,squared_loss,linear_loss)
    return huber_fn
model.compile(loss = create_huber, optimizer='nadam')

In [41]:
# 加载模型
model = keras.models.load_model('my_model_with_a_custom_loss.h5',custom_objects={'huber_fn':create_huber(2.0)})

In [42]:
class HuberLoss(keras.losses.Loss):
    def __init__(self, threshold = 1.0,**kwargs):
        self.threshold = threshold
        super().__init__(**kwargs)
    def call(self, y_true, y_pred):
        error = y_true-y_pred
        is_small_error = tf.abs(error) < self.threshold
        squared_loss = tf.square(error)/2
        linear_loss = self.threshold* tf.abs(error)-self.threshold**2/2
        return tf.where(is_small_error,squared_loss,linear_loss)
    def get_config(self):
        base_config = super().get_config()
        return {**base_config,'threshold':self.threshold}


---

## 一、构造函数 `__init__()`
### 核心要求
- 接收 `**kwargs`（关键字参数），并**透传给父类 `Loss` 的构造函数**
- 父类负责处理**标准超参数**：
  - `name`：损失函数的名称
  - `reduction`：单个实例损失的聚合算法

### `reduction` 超参数详解
| 取值 | 计算逻辑 | 说明 |
|------|----------|------|
| `sum_over_batch_size`（默认） | 实例损失 × 样本权重（如有） → 求和 → **除以批量大小** | 不是「权重平均」（分母是批量大小，不是权重之和） |
| `sum` | 实例损失 × 样本权重 → 直接求和 | 不做归一化，仅汇总总损失 |
| `none` | 不聚合，直接返回每个实例的原始损失 | 用于需要逐样本处理的场景 |

---

## 二、核心计算方法 `call()`
### 核心作用
- 输入：模型的**预测值 `y_pred`** 和真实**标签 `y_true`**
- 执行：计算**所有样本的实例损失**
- 输出：返回计算后的损失值（会自动按 `reduction` 规则聚合）


---

## 三、配置序列化方法 `get_config()`
### 核心作用
- 用于**模型保存/加载**，返回包含所有超参数的字典
- 保证损失类可以被Keras正确序列化，避免加载模型时报错

### 实现规范
1.  先调用**父类 `get_config()`**，获取父类的标准超参数（`name`、`reduction`）
2.  再将**自定义超参数**添加到字典中
3.  Python 3.5+ 推荐用 `{**父类字典, **自定义字典}` 语法合并字典（简洁高效）

---

## 四、关键注意事项
1.  **`reduction` 不要手动修改**：父类会自动处理聚合逻辑，`call()` 只需要返回逐样本损失即可
2.  **`**kwargs` 必须透传**：保证父类的标准超参数（`name`/`reduction`）正常生效
3.  **`get_config()` 必须实现**：否则模型保存后无法加载，是Keras自定义层/损失的强制要求
4.  **`sum_over_batch_size` 不是平均**：分母是批量大小，不是权重之和，和 `weighted_mean` 有本质区别


---

## 12.3.3 自定义激活函数，初始化，正则化和约束

In [43]:
def my_softplus(z):
    return tf.math.log(tf.exp(z)+1.0)

def my_glorot_initializer(shape,dtype=tf.float32):
    steddev =  tf.sqrt(2./(shape[0]*shape[1]))
    return tf.random.normal(shape,steddev,dtype=dtype)

def my_l1_regularizer(weights):
    return tf.reduce_sum(tf.abs(0.01*weights))

def my_positive_weights(weights):
    return tf.where(weights<0.,tf.zeros_like(weights),weights)

In [44]:
layer = keras.layers.Dense(30,activation=my_softplus,
                           kernel_initializer=my_glorot_initializer,
                           kernel_regularizer=my_l1_regularizer,
                           kernel_constraint=my_positive_weights)

In [45]:
# 保存factor超参数的l1正则化的简单类
class MyL1Regularizer(keras.regularizers.Regularizer):
    def __init__(self,factor):
        self.factor = factor
    def __call__(self, weights):
        return tf.reduce_sum(tf.abs(self.factor*weights))
    def get_config(self):
        return {'factor':self.factor}

## 12.3.4 自定义指标

- 损失被梯度下降用来训练模型，因此必须是可微的，梯度在任何地方都不应该为0，但是是否更容易被解释并不是重要的
- 而指标用于评估模型，必须更容易被理解，可以是不可微的或在各处有0梯度

- 在大多数情况下，自定义一个指标函数和一个损失函数完全相同

In [46]:
model.compile(loss = 'mse', optimizer = 'nadam',metrics=[create_huber(2.0)])

In [64]:
from tensorflow import keras
precision = keras.metrics.Precision()
precision([0,1,1,1,0,1,0,1],[1,1,0,1,0,1,0,1])

<tf.Tensor: shape=(), dtype=float32, numpy=0.800000011920929>

In [65]:
precision([0,1,0,0,1,0,1,1],[1,0,1,1,0,0,0,0])

<tf.Tensor: shape=(), dtype=float32, numpy=0.5>

In [66]:
precision.result()

<tf.Tensor: shape=(), dtype=float32, numpy=0.5>

In [67]:
# 用result获取指标的当前值
precision.variables

[<Variable path=precision_2/true_positives, shape=(1,), dtype=float32, value=[4.]>,
 <Variable path=precision_2/false_positives, shape=(1,), dtype=float32, value=[4.]>]

In [71]:
# 重置变量
precision.reset_state()

In [72]:
# 创建流式度量
class  HuberMetric(keras.metrics.Metric):
    def __init__(self,threshold = 1.0,**kwargs):
        super().__init__(**kwargs)
        self.threshold = threshold
        self.huber_fn = create_huber(threshold)
        self.total = self.add_weight('total',initializer = 'zeros')
        self.count = self.add_weight('count',initializer = 'zeros')
    def update_state(self,y_true,y_pred,sample_weight=None):
        metric = self.huber_fn(y_true,y_pred)
        self.total.assign_add(tf.reduce_sum(metric))
        self.count.assign_add(tf.cast(tf.size(y_true),tf.float32))
    def result(self):
        return self.total/self.count
    def get_config(self):
        base_config = super().get_config()
        return {**base_config,'threshold':self.threshold}



### 1. 构造函数（`__init__`）
- 使用 `add_weight()` 方法创建用于跟踪多个批次度量状态的变量。
  示例中包含了两个变量：
  - **总计**：所有 Huber 损失的总和
  - **计数**：到目前为止处理的实例数
- 也可以手动创建 `tf.Variable`，Keras 会自动跟踪任何设置为属性的可跟踪对象（如变量、层、模型）。

### 2. `update_state()` 方法
- 当将类的实例作为函数调用时（如 `metric_obj(y_true, y_pred)`），会自动调用此方法。
- 接收一批次的标签、预测值（以及可选的采样权重），用于更新内部变量（总计、计数等）。

### 3. `result()` 方法
- 计算并返回最终的度量结果，例如“所有实例的平均 Huber 损失”。
- 当度量对象被当作函数调用时，会先执行 `update_state()`，再执行 `result()`，并返回其输出。

### 4. `get_config()` 方法
- 用于确保度量中的参数（例如示例中的 `threshold`）能够随模型一起保存和加载。

### 5. `reset_states()` 方法
- 默认实现将所有变量重置为 0.0。
- 如有特殊需求，可以覆盖此方法自定义重置逻辑。


## 12.3.5 自定义层

In [73]:
# 要构建不带任何权重的自定义层，最简单的方法就是编写一个函数并将其包装在keras.layers.Lambda层中
exponential_layer = keras.layers.Lambda(lambda x: tf.exp(x))

In [75]:
# 要构建自定义的有状态层（具有权重的层），需要创建keras.layers.Layer类的子类
class MyDenes(keras.layers.Layer):
    def __init__(self,units,activation = None,**kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self,batch_input_shape):
        self.kernel = self.add_weight(
            name='kernel',shape=[batch_input_shape[-1],self.units],initializer = 'glorot_normal')
        self.bias = self.add_weight(
            name='bias',shape=[self.units],initializer = 'zeros'
        )
        super().build(batch_input_shape)
    def call(self,X):
        return self.activation(X @ self.kernel + self.bias)

    def get_config(self):
        base_config = super().get_config()
        return {**base_config,'units':self.units,'activation':keras.activations.serialize(self.activation)}


---

## 自定义全连接层 `MyDense` 的方法详解

### 1. `__init__` 构造函数
- **参数**：`units`（神经元数量）、`activation`（激活函数，可选）、`**kwargs`（接收标准层参数如 `name`、`trainable`、`input_shape` 等）。
- **操作**：
  - 调用 `super().__init__(**kwargs)` 将标准参数传给父类 `Layer`。
  - 保存 `units` 和 `activation` 为实例属性。
  - 使用 `keras.activations.get(activation)` 将激活参数转换为可调用的激活函数（支持字符串如 `"relu"`、函数或 `None`）。

### 2. `build(batch_input_shape)`
- **调用时机**：层首次使用时自动调用，此时 Keras 已知道输入形状。
- **作用**：创建可训练的权重变量（延迟构建）。
- **实现细节**：
  - 通过 `self.add_weight()` 创建 `kernel`（形状为 `[input_dim, units]`，其中 `input_dim = batch_input_shape[-1]`）。
  - 创建 `bias`（形状为 `[units]`）。
  - **最后必须调用** `super().build(batch_input_shape)` 将层标记为已构建（设置 `self.built = True`）。

### 3. `call(self, X)`
- **功能**：执行前向计算。
- **计算逻辑**：`activation(X @ kernel + bias)`
  即矩阵乘法、加偏置、应用激活函数。

### 4. `compute_output_shape(self, batch_input_shape)`
- **返回值**：`tf.TensorShape` 对象，表示该层输出的形状。
- **计算规则**：输出形状与输入形状相同，只是最后一个维度替换为 `self.units`。
  - 例如：`batch_input_shape` 为 `(None, 20)` → 输出形状 `(None, 10)`。
- **注意**：在 `tf.keras` 中，形状是 `tf.TensorShape` 实例，可使用 `as_list()` 转为 Python 列表。

### 5. `get_config(self)`
- **作用**：返回序列化配置字典，支持模型保存与加载。
- **实现**：
  - 获取父类配置：`base_config = super().get_config()`。
  - 合并自定义参数：`units` 和序列化的激活函数（`keras.activations.serialize(self.activation)`）。
- **结果**：保存时能够完全重建该层。

---

## 方法职责总览

| 方法 | 核心职责 |
|------|----------|
| `__init__` | 保存超参数，激活函数标准化，调用父类初始化 |
| `build` | 根据输入形状创建权重（kernel, bias） |
| `call` | 实现前向计算（线性变换 + 激活） |
| `compute_output_shape` | 返回输出张量的形状 |
| `get_config` | 提供序列化配置，支持模型保存/加载 |

遵循这个模板，可以创建任意自定义的、带有可训练权重的 Keras 层。

In [76]:
# 创建具有多个输入的层，call方法的参数应该是包含所有输入的元组
# 两个输入，三个输出
class MyMultilayer(keras.layers.Layer):
    def call(self,X):
        X1,X2 = X
        return [X1+X2,X1*X2,X1/X2]

    def compute_output_shape(self, batch_input_shape):
        b1,b2 = batch_input_shape
        return [b1,b1,b1]


In [77]:
# 创建一个在训练期间（用于正则化）添加高斯噪声但在测试期间不执行任何操作的层
class MyGaussianNoise(keras.layers.Layer):
    def __init__(self,stddev,**kwargs):
        super().__init__(**kwargs)
        self.stddev = stddev

    def call(self,X,training=None):
        if training:
            noise = tf.random.normal(tf.shape(X),stddev=self.stddev)
            return X + noise
        else:
            return X
    def compute_output_shape(self, batch_input_shape):
        return batch_input_shape

## 12.3.6 自定义模型

In [78]:
#该模型无太大意义仅仅是用作示例
class ResidualBlock(keras.layers.Layer):
    def __init__(self,n_layers,n_neurons,**kwargs):
        super().__init__(**kwargs)
        self.hidden = [keras.layers.Dense(units=n_neurons,activation='elu',kernel_initializer = 'he_normal') for _ in range(n_layers)]

    def call(self,inputs):
        Z = inputs
        for layer in self.hidden:
            Z = layer(Z)
        return inputs+Z


In [79]:
# 使用子类API定义模型本身
class ResidualRegressor(keras.Model):
    def __init__(self,output_dim,**kwargs):
        super().__init__(**kwargs)
        self.hidden1 = keras.layers.Dense(30,activation='elu',kernel_initializer = 'he_normal')

        self.block1  =ResidualBlock(2,30)
        self.block2 = ResidualBlock(2,30)
        self.out  =keras.layers.Dense(output_dim)

    def call(self,inputs):
        Z = self.hidden1(inputs)
        for _ in range(1+3):
            Z = self.block1(Z)
        Z =self.block2(Z)
        return self.out(Z)

## 12.3.7 基于模型内部的损失和指标

In [80]:
#带有自定义重建损失的自定义模型的代码
class ReconstructingRegressor(keras.Model):
    def __init__(self,output_dim,**kwargs):
        super().__init__(**kwargs)
        self.hidden = [keras.layers.Dense(30,activation='selu',kernel_initializer = 'lecun_normal') for _ in range(5)]
        self.out = keras.layers.Dense(output_dim)

    def build(self,batch_input_shape):
        n_inputs = batch_input_shape[-1]
        self.reconstruct = keras.layers.Dense(n_inputs)
        super().build(batch_input_shape)

    def call(self,inputs):
        Z = inputs
        for layer in self.hidden:
            Z = layer(Z)
        reconstruction = self.reconstruct(Z)
        recon_loss = tf.reduce_mean(tf.square(reconstruction-inputs))
        self.add_loss(0.05*recon_loss)
        return self.out(Z)


---

## 需求拆解
这是一个**带自监督重建损失的自定义回归模型**，核心要求如下：
1.  **构造函数 (`__init__`)**：创建包含 **5 个密集隐藏层** + **1 个密集输出层** 的深度神经网络（DNN）
2.  **`build()` 方法**：创建一个**重建层**（Dense层），用于还原输入数据。必须在`build`中创建，因为输入维度只有在调用`build`时才确定
3.  **`call()` 方法**：
    - 让输入依次通过5个隐藏层
    - 将隐藏层输出传入重建层，生成输入的重构结果
    - 计算**重建损失**（重构值与原始输入的均方误差MSE），乘以0.05缩放后，用`add_loss()`加入模型总损失
    - 将隐藏层输出传入输出层，返回最终回归预测结果
4.  **核心设计**：用**重建损失作为正则项**，约束隐藏层学习到有意义的特征，同时不影响主回归任务的性能



---

## 逐模块深度解析
### 1. `__init__` 构造函数
| 代码部分 | 作用详解 |
|---------|---------|
| `class ReconstructingRegressor(keras.Model)` | 继承Keras的`Model`类，实现完整可训练模型，支持`fit()`/`predict()`等原生方法 |
| `self.hidden = [Dense(30, activation='selu', kernel_initializer='lecun_normal') for _ in range(5)]` | 创建5个全连接隐藏层：<br>- `30`：每层30个神经元<br>- `activation='selu'`：自归一化激活函数，无需BatchNorm即可稳定训练<br>- `kernel_initializer='lecun_normal'`：专门适配selu的初始化，保证输入输出方差一致 |
| `self.out = Dense(output_dim)` | 输出层，将30维隐藏特征映射到回归目标维度，默认无激活函数（回归任务标准设置） |

---

### 2. `build` 方法
| 代码部分 | 作用详解 |
|---------|---------|
| `def build(self, batch_input_shape)` | Keras自定义模型的标准方法，**仅在模型第一次接收输入时自动调用一次**，用于创建依赖输入形状的层 |
| `n_inputs = batch_input_shape[-1]` | 提取输入特征的维度（输入张量最后一维），这个值在`__init__`阶段是未知的，因此必须在`build`中获取 |
| `self.reconstruct = Dense(n_inputs)` | 创建重建层：神经元数等于输入特征数，作用是**把隐藏层的30维特征还原成原始输入**，强制隐藏层学习到能重构输入的有效特征 |
| `super().build(batch_input_shape)` | 调用父类`Model`的`build`方法，标记模型为"已构建"，是Keras自定义层/模型的规范写法 |

---

### 3. `call` 方法（核心逻辑）
| 代码部分 | 作用详解 |
|---------|---------|
| `Z = inputs; for layer in self.hidden: Z = layer(Z)` | 输入依次通过5个隐藏层，完成特征提取，得到30维隐藏表示`Z` |
| `reconstruction = self.reconstruct(Z)` | 用重建层把30维隐藏特征`Z`还原成和原始输入维度一致的重构值 |
| `recon_loss = tf.reduce_mean(tf.square(reconstruction - inputs))` | 计算**重建损失**：均方误差（MSE），衡量重构值和原始输入的差异 |
| `self.add_loss(0.05 * recon_loss)` | ✅ 核心设计：<br>- `add_loss()`：将自定义损失加入模型总损失，无需在`compile()`中指定<br>- `0.05`：缩放系数，控制重建损失的权重，避免其主导主回归任务的损失（超参数可调整） |
| `return self.out(Z)` | 将隐藏特征`Z`传入输出层，返回最终的回归预测结果 |

---

## 核心原理：为什么要加重建损失？
### 1. 本质：自监督正则化
- 传统回归模型只优化**预测值 vs 真实标签**的监督损失，容易过拟合，尤其是小数据集
- 加入**重建损失**后，模型需要同时满足两个目标：
  1.  主任务：准确预测回归标签（监督学习）
  2.  辅助任务：用隐藏特征还原原始输入（自监督学习）
- 强制隐藏层学习到**能保留输入信息的有效特征**，相当于给模型加了一层正则化，提升泛化能力

### 2. 关键设计细节
| 设计点 | 原因 |
|--------|------|
| 用`selu`+`lecun_normal` | selu是自归一化激活函数，配合lecun_normal初始化，深层网络无需BatchNorm即可保持训练稳定，简化结构 |
| 重建层放在`build`中 | 输入维度只有在模型第一次接收数据时才确定，`__init__`阶段无法获取，因此必须在`build`中创建 |
| 损失乘以0.05 | 控制重建损失的权重，避免其主导主任务损失，让模型优先优化回归性能，同时用重建损失做正则 |
| 用`add_loss()` | 无需修改模型编译时的损失函数，直接将自定义损失加入总损失，不影响主任务的训练流程 |


---



## 12.3.8 使用自动微分计算梯度

In [81]:
def f(w1,w2):
    return 3*w1**2+2*w1*w2

In [82]:
w1,w2 = 5,3
eps = 1e-6
(f(w1+eps,w2) - f(w1, w2))/eps


36.000003007075065

In [83]:
(f(w1,w2+eps) - f(w1, w2))/eps

10.000000003174137

In [84]:
#自动微分
w1,w2 = tf.Variable(5.),tf.Variable(3.)
with tf.GradientTape() as tape:
    z = f(w1,w2)

gradients = tape.gradient(z,[w1,w2])

In [85]:
gradients

[<tf.Tensor: shape=(), dtype=float32, numpy=36.0>,
 <tf.Tensor: shape=(), dtype=float32, numpy=10.0>]

gradient()方法都只经历一次已经记录的计算（以相反的顺序）

In [86]:
# 如果尝试两次调用gradient()会报错
with tf.GradientTape as tape:
    z = f(w1,w2)

dz_w1= tape.gradient(z, w1)
dz_w2 =tape.gradient(z, w2)# 报错

AttributeError: __enter__

In [87]:
# 所以每次使用完tape后将其删除以释放资源
with tf.GradientTape(persistent=True) as tape:
    z = f(w1,w2)

dz_w1 = tape.gradient(z,w1)
dz_w2 = tape.gradient(z,w2)
del tape

In [90]:
# 默认情况下，tape只追踪涉及变量的操作，因此如果尝试针对变量以外的任何其他变量计算z的梯度结果将为None
c1,c2 = tf.constant(5.),tf.constant(3.)
with tf.GradientTape() as tape:
    z = f(c1,c2)

gradients = tape.gradient(z, [c1,c2])

In [91]:
# 强制tape观察变量
with tf.GradientTape() as tape:
    tape.watch(c1)
    tape.watch(c2)
    z = f(c1,c2)

gradients = tape.gradient(z, [c1,c2])

In [92]:
# 阻止梯度在某些部分的反向传播
def f(w1,w2):
    return 3*w1**2+tf.stop_gradient(2*w1*w2)

with tf.GradientTape() as tape:
    z = f(w1,w2)

gradients = tape.gradient(z,[w1,w2])

In [93]:
# 计算梯度时，有可能会遇到一些数值问题
x = tf.Variable([100.])
with tf.GradientTape() as tape:
    z = my_softplus(x)

tape.gradient(z,[x])

[<tf.Tensor: shape=(1,), dtype=float32, numpy=array([nan], dtype=float32)>]

In [94]:
#修饰my_softplus()使它即返回其正常输出又返回计算导数的函数
@tf.custom_gradient
def my_better_softplus(z):
    exp = tf.exp(z)
    def my_softplus_gradients(grad):
        return grad / (1+1/exp)
    return tf.math.log(exp+1),my_softplus_gradients

## 12.3.9 自定义训练循环

In [110]:
# 首先建立一个简单的模型
l2_reg = keras.regularizers.l2(0.05)
model = keras.models.Sequential([
    keras.layers.Dense(30,activation='elu',kernel_initializer = 'he_normal',kernel_regularizer=l2_reg),
    keras.layers.Dense(1,kernel_regularizer = l2_reg)
])

In [111]:
# 从训练集中随机采样一批实例
def random_batch(X,y,batch_size = 32):
    idx = np.random.randint(len(X),size=batch_size)
    return X[idx],y[idx]

In [118]:
# 定义函数来显示训练状态，包括步数，步总数，从轮次开始以来的平均损失和其他指标
def print_status_bar(iteration,total,loss,metrics = None):
    metrics = '-'.join(["{}:{:.4f}".format(m.name,m.result()) for m in [loss]+(metrics or [])])
    end = "" if iteration <total else '\n'
    print('\r{}/{} - '.format(iteration,total)+metrics,end=end)

In [119]:
import tensorflow as tf
from tensorflow import keras
# 定义一些超参数
n_epochs = 5
batch_size = 32
n_steps = len(X_train)//batch_size
optimizer = keras.optimizers.Nadam(learning_rate=0.01)
loss_fn = keras.losses.MeanSquaredError()
mean_loss = keras.metrics.Mean()
metrics = [keras.metrics.MeanAbsoluteError()]
l2_reg = keras.regularizers.L2(0.05)

In [120]:
#构建自定义循环
for epoch in range(1,n_epochs+1):
    print('Epoch {}/{}'.format(epoch,n_epochs))
    for step in range(1,n_steps+1):
        X_batch,y_batch = random_batch(X_train_scaled,y_train)
        with tf.GradientTape() as tape:
            y_pred = model(X_batch,training = True)
            main_loss = tf.reduce_mean(loss_fn(y_batch,y_pred))
            loss = tf.add_n([main_loss]+model.losses)
        gradients = tape.gradient(loss,model.trainable_variables)
        optimizer.apply_gradients(zip(gradients,model.trainable_variables))
        mean_loss(loss)
        for metric in metrics:
            metric(y_batch,y_pred)
        print_status_bar(step*batch_size,len(y_train),mean_loss,metrics)
    print_status_bar(len(y_train),len(y_train),mean_loss,metrics)
    for metric in[mean_loss] + metrics:
        metric.reset_state()

Epoch 1/5
1000/1000 - mean:5.9396-mean_absolute_error:0.9649
Epoch 2/5
1000/1000 - mean:3.8784-mean_absolute_error:0.5424
Epoch 3/5
1000/1000 - mean:2.9826-mean_absolute_error:0.4350
Epoch 4/5
1000/1000 - mean:2.4314-mean_absolute_error:0.4599
Epoch 5/5
1000/1000 - mean:1.7945-mean_absolute_error:0.3335




1. **计算批次均值**
   - 使用 `tf.reduce_mean()` 计算当前批次样本的均值。
   - 如需对每个样本赋予不同权重，可在此处进行自定义调整。

2. **处理正则化损失**
   - 正则化损失已被归约到单个标量值。
   - 使用 `tf.add_n()` 对多个形状、数据类型相同的张量求和，得到总正则化损失。

3. **计算梯度并更新参数**
   - 利用 `tape` 计算损失相对于**每个可训练变量**（而非所有变量）的梯度。
   - 调用优化器的“梯度下降”步骤，根据梯度更新模型参数。

4. **更新并显示训练状态**
   - 更新当前训练轮次（epoch）内的平均损失和评估指标。
   - 在训练过程中实时显示进度条（状态栏）及相关数值。

5. **重置状态**
   - 在每轮训练结束后，重置平均损失和指标的状态，为下一轮训练做好准备。

---

In [121]:
# 对模型添加权重约束
for variable in model.variables:
    if variable.constraint is not None:
        variable.assign(variable.constraint(variable))

# 12.4 TensorFlow函数和图

In [122]:
def cube(x):
    return x**3

In [123]:
cube(2)

8

In [124]:
cube(tf.constant(2.))

<tf.Tensor: shape=(), dtype=float32, numpy=8.0>

In [125]:
# 将python函数转换为TensorFlow函数
tf_cube = tf.function(cube)
tf_cube

In [126]:
# 作为修饰器
@tf.function
def tf_cube(x):
    return x**3


In [127]:
# 使用原python函数
tf_cube.python_function(2)

8

TF函数一般比原始的python函数更快，如果想增强python函数，那么就只需要将其转换为TF函数

## 12.4.1 自动图和跟踪


---

## 一、核心目标：TensorFlow 如何生成计算图？
TensorFlow 要把**普通 Python 函数**转换成可执行的计算图（Graph），需要两步核心操作：**自动图（AutoGraph）** + **跟踪（Tracing）**。

---

## 二、第一步：自动图（AutoGraph）——把 Python 语法转成 TF 操作
### 1. 为什么需要 AutoGraph？
Python 原生语法（`for`/`while` 循环、`if` 分支、`break`/`continue`/`return` 等）没有对应的底层接口，无法直接被 TensorFlow 捕获。
AutoGraph 的作用就是：**分析 Python 函数的源代码，把原生控制流语句，转换成等价的 TensorFlow 图操作**。

### 2. 转换规则
| Python 原生语法 | 转换后的 TensorFlow 操作 |
|----------------|--------------------------|
| `+`、`*` 等运算符 | `tf.__add__()`、`tf.__mul__()` |
| `if` 条件分支 | `tf.cond()` |
| `for`/`while` 循环 | `tf.while_loop()` |
| `break`/`continue`/`return` | 对应循环/分支的控制逻辑 |

### 3. 示例：`sum_squares()` 函数的转换
以计算平方和的函数为例：
```python
# 原始 Python 函数
def sum_squares(n):
    s = 0
    for i in tf.range(n + 1):
        s += i ** 2
    return s
```
AutoGraph 会做两件事：
1.  提取 `for` 循环的主体 `loop_body(i, s)`（即 `s += i ** 2`）
2.  用 `ag__.for_stmt()` 包装循环，最终生成等价的 `tf.while_loop()` 操作
3.  输出一个「升级版」函数 `tf_sum_squares()`，完全用 TF 操作实现，可被图执行

---

## 三、第二步：跟踪（Tracing）——用符号张量生成最终计算图
### 1. 什么是跟踪？
AutoGraph 生成升级函数后，TensorFlow 会用**符号张量（Symbolic Tensor）**调用这个函数，完成最终计算图的构建。
- 符号张量：**没有实际值的「空张量」**，仅包含「名称、数据类型、形状」三个属性
- 跟踪过程：在「图模式」下运行函数，只记录操作，不执行实际计算（和 TensorFlow 1.x 的默认图模式一致）

### 2. 跟踪的核心逻辑
1.  用符号张量（比如 `shape=[]`、`dtype=int32` 的张量）调用 `tf_sum_squares()`
2.  函数在图模式下执行，每一个 TF 操作都会在图中添加一个节点，输出张量作为边
3.  最终生成完整的计算图：节点代表操作，箭头代表张量流动（如图 12-4 所示）

### 3. 两种执行模式对比
| 模式 | 特点 | 作用 |
|------|------|------|
| **图模式（Graph Mode）** | 不执行实际计算，只记录操作，生成计算图 | 跟踪阶段，用于构建图 |
| **即时执行（Eager Mode）** | 立即执行计算，返回实际值 | 模型训练/推理阶段，用于实际运算 |

---

## 四、完整工作流程（两步闭环）
```
原始 Python 函数 → 【AutoGraph 自动转换】 → 升级版 TF 函数 → 【Tracing 符号跟踪】 → 最终计算图
```
对应图 12-4 的两个阶段：
1.  **自动图（AutoGraph）**：分析 `sum_squares()` 源码，把 `for` 循环转成 `tf.while_loop()`，生成 `tf_sum_squares()`
2.  **跟踪（Tracing）**：用符号张量调用 `tf_sum_squares()`，记录所有操作，生成最终的计算图结构

---

## 五、补充：查看生成的图代码
如果需要调试、查看 AutoGraph 生成的升级函数源码，可以调用：
```python
tf.autograph.to_code(sum_squares.python_function)
```
生成的代码格式会比较冗余，但可以清晰看到 Python 语法被转换成 TF 操作的过程，用于调试。

---

## 六、核心知识点总结（记忆版）
1.  **AutoGraph 核心**：把 Python 原生控制流（循环、分支）转换成 TensorFlow 可识别的图操作，解决 Python 语法无法被捕获的问题。
2.  **Tracing 核心**：用符号张量运行升级函数，只记录操作、不计算值，生成最终的计算图。
3.  **符号张量本质**：无实际值，仅存「名、类型、形状」，是构建计算图的「占位符」。
4.  **图模式 vs 即时执行**：图模式用于建图，即时执行用于实际运算，是 TensorFlow 2.x 图执行的核心逻辑。

---

## 七、一句话总结
12.4.1 讲的是 TensorFlow 2.x 图执行的核心原理：**先通过 AutoGraph 把 Python 函数转成 TF 操作，再用 Tracing 跟踪符号张量，最终生成可执行的计算图**，让普通 Python 代码能享受图模式的性能优化。

---

## 12.4.2 TF函数规则


---

## 一、核心前提：什么是 TF 函数？
TF 函数（TF Function）是 TensorFlow 2.x 的核心特性：
- 用 `@tf.function` 装饰普通 Python 函数，就能把它转换成**可在图模式下执行的计算图函数**
- 核心优势：自动优化、加速执行、支持部署到不同平台
- 但要让转换顺利，必须遵守一套「TF 函数规则」，否则会出现逻辑错误、性能问题

---

## 二、核心规则 1：只能用 TensorFlow 原生操作，禁止外部库/原生 Python 操作
### 1. 规则本质
TensorFlow 的计算图**只能包含 TensorFlow 原生构造**：张量、运算、变量。
- ❌ 禁止：NumPy、Python 标准库等外部库的操作
- ✅ 必须：用 TensorFlow 等价操作替代

### 2. 为什么？
外部库操作**只会在「跟踪（Tracing）阶段」运行一次**，不会被写入计算图。
- 跟踪阶段：用符号张量建图，仅运行一次
- 图执行阶段：仅运行图内的 TF 操作，外部操作不会再执行
- 后果：外部操作的结果会被「硬编码」进图，后续调用不会更新，导致逻辑错误

### 3. 典型示例与替代
| 错误写法（外部库） | 正确写法（TF 原生） |
|-------------------|---------------------|
| `np.sum(x)`        | `tf.reduce_sum(x)`  |
| `sorted(x)`        | `tf.sort(x)`         |
| `np.random.rand()` | `tf.random.uniform([])` |

### 4. 关键坑点：随机数生成
- ❌ 错误：用 `np.random.rand()` 生成随机数
  - 仅在跟踪时生成一次随机数，后续所有调用都返回同一个值
- ✅ 正确：用 `tf.random.uniform([])` 生成随机数
  - 操作会被写入图，每次调用都生成新的随机数

---

## 三、核心规则 2：禁止在 TF 函数中使用有副作用的非 TF 代码
### 1. 规则本质
非 TensorFlow 代码的**副作用**（如打印日志、更新 Python 计数器、修改全局变量等），**只会在跟踪阶段执行一次**，不会在图执行阶段重复执行。
- 后果：你期望每次调用都执行的操作，实际只会执行一次，逻辑完全错误

### 2. 典型错误示例
```python
# 错误示例：计数器只会在跟踪时+1，后续调用不会更新
counter = 0
@tf.function
def my_func(x):
    global counter
    counter += 1  # 仅跟踪时执行一次，图执行时不会执行
    return x + 1
```
- 调用 100 次 `my_func`，`counter` 只会变成 1，而不是 100

---

## 四、核心规则 3：`tf.py_function`：包装纯 Python 代码的「双刃剑」
### 1. 作用
可以用 `tf.py_function` 把任意 Python 代码包装成 TF 操作，强行加入计算图。

### 2. 代价（必须注意）
- ❌ 性能大幅降低：TensorFlow 无法对这部分代码做图优化
- ❌ 可移植性变差：只能在安装了对应 Python 环境的平台运行，无法部署到 TPU、移动端等
- ❌ 图完整性破坏：这部分代码脱离 TF 图的优化体系，破坏了图的一致性

### 3. 使用建议
**除非万不得已，否则绝对不要用**。仅在无法用 TF 操作实现的特殊场景下使用。

---

## 五、核心规则 4：递归、Python 函数等也需遵守相同规则
- 即使是 TF 函数调用的其他 Python 函数、递归函数，也必须遵守上述所有规则
- 所有被 `@tf.function` 修饰的函数，其内部的所有操作，都必须符合 TF 函数的要求
- 否则，同样会出现「仅跟踪执行、图内不生效」的问题

---

## 六、补充：查看生成的 TF 函数源码（调试用）
和 12.4.1 一致，可以用以下命令查看 AutoGraph 生成的 TF 函数源码，用于调试：
```python
tf.autograph.to_code(your_tf_function.python_function)
```
生成的代码格式冗余，但能清晰看到转换过程，排查规则违反问题。

---

## 七、核心知识点总结（记忆版）
| 规则 | 核心要点 | 避坑关键 |
|------|----------|----------|
| 仅用 TF 原生操作 | 图只能包含 TF 张量/运算/变量，禁止 NumPy/标准库 | 用 TF 操作替代所有外部库操作，尤其是随机数 |
| 禁止有副作用的非 TF 代码 | 副作用仅在跟踪时执行一次，图内不生效 | 不要在 TF 函数里修改 Python 全局变量、打印日志 |
| `tf.py_function` 慎用 | 可包装 Python 代码，但牺牲性能和可移植性 | 非必要不使用，优先用 TF 原生实现 |
| 所有嵌套函数遵守规则 | 递归、调用的子函数也必须符合要求 | 全链路都要遵循 TF 函数规则，不能有遗漏 |

---

## 八、一句话总结
12.4.2 讲的是**把 Python 函数转换成 TF 函数必须遵守的规则**：核心是「只能用 TF 原生操作，禁止外部库和有副作用的代码」，否则会出现逻辑错误、性能问题，确保计算图的正确性和可优化性。

---
